In [ ]:
from flask import Flask, jsonify, request
import yfinance as yf
import threading
from getpass import getpass
from google import genai

# -------------------------------
# Ask for Gemini API Key at startup
# -------------------------------
gemini_key = getpass("Enter your Gemini API Key: ")
client = genai.Client(api_key=gemini_key)

# -------------------------------
# Create Flask app
# -------------------------------
app = Flask(__name__)

# -------------------------------
# Company & Stock Info Endpoint
# -------------------------------
@app.route("/api/company/<symbol>", methods=["GET"])
def company_info(symbol):
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info

        # Company info variables
        company_name = info.get("longName", "N/A")
        industry = info.get("industry", "N/A")
        sector = info.get("sector", "N/A")
        summary = info.get("longBusinessSummary", "N/A")
        website = info.get("website", "N/A")
        full_time_employees = info.get("fullTimeEmployees", None)

        officers = info.get("companyOfficers", [])
        key_officers = [{"name": o.get("name"), "title": o.get("title")} for o in officers]

        # Real-time stock data variables
        current_price = info.get("currentPrice", None)
        previous_close = info.get("previousClose", None)
        day_low = info.get("dayLow", None)
        day_high = info.get("dayHigh", None)
        volume = info.get("volume", None)
        market_cap = info.get("marketCap", None)

        if current_price and previous_close:
            price_change = current_price - previous_close
            percent_change = (price_change / previous_close) * 100
        else:
            price_change = percent_change = None

        # Historical data
        historical_df = ticker.history(period="1mo")  # last 1 month
        historical_data = historical_df.reset_index().to_dict(orient="records")

        # Gemini LLM summary
        llm_prompt = f"""Perform a comprehensive analysis for {company_name} and deliver actionable insightsusing the following data: industry={industry}, sector={sector}, full_time_employees={full_time_employees}
        current_price={current_price}, previous_close={previous_close}, day_low={day_low}, day_high={day_high}, volume={volume}, market_cap={market_cap}
        price_change={price_change}, percent_change={percent_change}, historical_data={historical_data}
        """
        llm_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=llm_prompt
        )
        company_summary_llm = llm_response.text

        # Return all variables in JSON
        data = {
            "symbol": symbol.upper(),
            "company_name": company_name,
            "industry": industry,
            "sector": sector,
            "website": website,
            "full_time_employees": full_time_employees,
            "key_officers": key_officers,
            "current_price": current_price,
            "previous_close": previous_close,
            "price_change": price_change,
            "percent_change": percent_change,
            "day_low": day_low,
            "day_high": day_high,
            "volume": volume,
            "market_cap": market_cap,
            "long_summary": summary,
            "llm_summary": company_summary_llm,
            "historical_data": historical_data
        }

        return jsonify(data), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# -------------------------------
# Run Flask in background
# -------------------------------
def run_app():
    app.run(port=5000, debug=True, use_reloader=False)


thread = threading.Thread(target=run_app)
thread.start()

# Now you can access: http://127.0.0.1:5000/api/company/AAPL